In [20]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_etl = config['ETL_PROCESS']

url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")

etl_conn = create_engine(url_etl)

In [21]:
p1 = pd.read_sql_query("""
    SELECT 
        f.mes, 
        COUNT(*) AS total_servicios
    FROM fact_servicio h
    JOIN dim_fechahora f ON h.fk_fecha_iniciado = f.fecha_hora_key
    WHERE h.fk_fecha_iniciado <> -1
    GROUP BY f.mes
    ORDER BY total_servicios DESC;
""", etl_conn)
p1

,mes,total_servicios
0,5.0,4725
1,7.0,4549
2,4.0,4480
3,8.0,4304
4,6.0,4184
5,3.0,3337
6,2.0,2479
7,1.0,296
8,12.0,25
9,9.0,21


In [22]:
p2 = pd.read_sql_query("""
    SELECT 
        f.dia_de_la_semana, 
        COUNT(*) AS total_solicitudes
    FROM fact_servicio h
    JOIN dim_fechahora f ON h.fk_fecha_iniciado = f.fecha_hora_key
    WHERE h.fk_fecha_iniciado <> -1
    GROUP BY f.dia_de_la_semana
    ORDER BY total_solicitudes DESC;
""", etl_conn)
p2

,dia_de_la_semana,total_solicitudes
0,Tuesday,5398
1,Friday,5281
2,Thursday,5160
3,Wednesday,4961
4,Monday,4308
5,Saturday,2481
6,Sunday,840


In [23]:
p3 = pd.read_sql_query("""
    SELECT 
        f.hora, 
        COUNT(*) AS servicios_recogidos
    FROM fact_servicio h
    JOIN dim_fechahora f ON h.fk_fecha_recogido = f.fecha_hora_key
    WHERE h.fk_fecha_recogido <> -1
    GROUP BY f.hora
    ORDER BY servicios_recogidos DESC;
""", etl_conn)
p3

,hora,servicios_recogidos
0,10.0,3113
1,11.0,3004
2,9.0,2925
3,15.0,2804
4,8.0,2781
5,16.0,2365
6,14.0,2296
7,12.0,2230
8,17.0,1250
9,13.0,1213


In [24]:
p4 = pd.read_sql_query("""
    SELECT 
        f.año, 
        f.mes, 
        c.nombre_cliente, 
        COUNT(*) AS total_servicios
    FROM fact_servicio h
    JOIN dim_cliente c ON h.fk_cliente = c.cliente_key
    JOIN dim_fechahora f ON h.fk_fecha_iniciado = f.fecha_hora_key
    WHERE h.fk_fecha_iniciado <> -1 AND h.fk_cliente <> -1
    GROUP BY f.año, f.mes, c.nombre_cliente
    ORDER BY f.año, f.mes, total_servicios DESC;
""", etl_conn)
p4

,año,mes,nombre_cliente,total_servicios
0,2023.0,9.0,CLINICA CALI -JOVEN,18
1,2023.0,9.0,CLINICA DEPORTIVA DEL SUR,3
2,2023.0,10.0,CLINICA CALI -JOVEN,12
3,2023.0,11.0,CLINICA CALI -JOVEN,16
4,2023.0,11.0,CRUZ AZUL-LIFE,1
...,...,...,...,...
102,2024.0,8.0,FUNDACION CLINICA INFANTIL LOS GRITONES,19
103,2024.0,8.0,CRUZ AZUL-LIFE,16
104,2024.0,8.0,CLINICA SAN RAFAEL DE OCCIDENTE,3
105,2024.0,8.0,Cliente 1,2


In [25]:
p5 = pd.read_sql_query("""
    SELECT 
        m.nombre_completo, 
        COUNT(*) AS servicios_entregados
    FROM fact_servicio h
    JOIN dim_mensajero m ON h.fk_mensajero = m.mensajero_key
    WHERE h.fk_mensajero <> -1 AND h.fk_fecha_entregado <> -1
    GROUP BY m.nombre_completo
    ORDER BY servicios_entregados DESC;
""", etl_conn)
p5

,nombre_completo,servicios_entregados
0,JHONTROCHEZ,2429
1,Sebastian Acuña,1505
2,Juan Solano,1465
3,ANDRESGUTIERREZ,1425
4,Jan Sastre,1309
5,Hector Aquiles,1304
6,LUISCARDONA,1304
7,JPEDROZA,1231
8,Jairon Montes,1214
9,JHONMUÑOZ,1190


In [ ]:
p6_por_año = pd.read_sql_query("""
SELECT
    f.año,
    c.nombre_cliente,
    s.nombre AS sede,
    COUNT(*) AS total_servicios
FROM fact_servicio h
JOIN dim_cliente c ON h.fk_cliente = c.cliente_key
JOIN dim_sede s ON h.fk_sede = s.sede_key
JOIN dim_fechahora f ON h.fk_fecha_iniciado = f.fecha_hora_key
WHERE h.fk_cliente <> -1 AND h.fk_sede <> -1 AND h.fk_fecha_iniciado <> -1
GROUP BY f.año, c.nombre_cliente, s.nombre
ORDER BY f.año, c.nombre_cliente, total_servicios DESC;
""", etl_conn)
p6_por_año

,año,nombre_cliente,sede,total_servicios
0,2023.0,CARROS DEL PACIFICO (CHINA),PRINCIPAL NORTE / CHINA PACIFICO,3
1,2023.0,CARROS DEL PACIFICO (CHINA),FORDES PASOANCHOS,1
2,2023.0,CARROS DEL PACIFICO (CHINA),FORD CAÑAS FLASCAS,1
3,2023.0,CLINICA CALI -JOVEN,sede aux - cliente 1,51
4,2023.0,CLINICA CALI -JOVEN,VASQUEZ COBO,7
5,2023.0,CLINICA CALI -JOVEN,PRINCIPAL / OCCIDENTE,4
6,2023.0,CLINICA CALI -JOVEN,CAPITOLIO,1
7,2023.0,CLINICA CALI -JOVEN,INGENIO,1
8,2023.0,CLINICA CALI -JOVEN,VILLACOLOMBIA,1
9,2023.0,CLINICA DEPORTIVA DEL SUR,PRINCIPAL /14,3


In [35]:
p7_por_año = pd.read_sql_query("""
    SELECT
        f.año,
        ROUND(AVG(h.duracion_total_min)::numeric, 2) AS promedio_minutos,
        ROUND((AVG(h.duracion_total_min) / 60)::numeric, 2) AS promedio_horas
    FROM fact_servicio h
    JOIN dim_fechahora f ON h.fk_fecha_iniciado = f.fecha_hora_key
    WHERE h.fk_fecha_cerrado <> -1                           
    GROUP BY f.año
    ORDER BY f.año;
""", etl_conn)
p7_por_año

,año,promedio_minutos,promedio_horas
0,2023.0,82641.53,1377.36
1,2024.0,541.59,9.03


In [38]:
p8_por_año = pd.read_sql_query("""
    SELECT
        f.año,
        ROUND(AVG(h.duracion_iniciado_asignado_min)::numeric, 2) AS asignacion,
        ROUND(AVG(h.duracion_asignado_recogido_min)::numeric, 2) AS recogida,
        ROUND(AVG(h.duracion_recogido_entregado_min)::numeric, 2) AS entrega,
        ROUND(AVG(h.duracion_entregado_cerrado_min)::numeric, 2) AS cierre
    FROM fact_servicio h
    JOIN dim_fechahora f ON h.fk_fecha_iniciado = f.fecha_hora_key
    WHERE h.fk_fecha_cerrado <> -1
    GROUP BY f.año
    ORDER BY f.año;
""", etl_conn)
p8_por_año

,año,asignacion,recogida,entrega,cierre
0,2023.0,36656.41,36363.34,7426.57,2195.20
1,2024.0,108.64,64.91,78.09,289.98


In [39]:
p9_por_año = pd.read_sql_query("""
    SELECT
        f.año,
        dn.nombre_novedad,
        COUNT(*) AS total_ocurrencias
    FROM fact_novedad h
    JOIN dim_novedad dn ON h.fk_tipo_novedad = dn.novedad_key
    JOIN dim_fechahora f ON h.fk_fecha = f.fecha_hora_key
    WHERE h.fk_tipo_novedad <> -1 AND h.fk_fecha <> -1
    GROUP BY f.año, dn.nombre_novedad
    ORDER BY f.año, total_ocurrencias DESC;
""", etl_conn)
p9_por_año

,año,nombre_novedad,total_ocurrencias
0,2023.0,Novedades del servicio,9
1,2023.0,No puedo continuar,8
2,2024.0,Novedades del servicio,3883
3,2024.0,No puedo continuar,1308


In [21]:
print("Usuario:", config_etl['user'])

Usuario: daniel
